# Visual encoder training — v6

**Native notebook training.** Setup, data, optimization, evaluation and saving run in this kernel; exceptions show full notebook tracebacks. No training subprocess, CLI or hidden working-directory dependency. RL training remains local and unchanged.

Use a GPU runtime. Upload `train-256-rollouts.pt`, `val-256-rollouts.pt`, and `real-256.pt` to `My Drive/triage-data/`. Run top to bottom. Start with batch 2; increase only after measuring memory. No claim that a particular batch fits every GPU.

Model/teacher code is downloaded at a pinned revision; training code lives below. Restart the runtime before changing the model-code revision. Internet is needed for dependencies, source and teacher weights.


In [ ]:
%pip install -q omegaconf huggingface_hub pillow
import sys, json, time, shutil, urllib.request
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
assert torch.cuda.is_available(), "Select a GPU runtime before training."
ROOT = Path('/content/triage-v6')
ROOT.mkdir(parents=True, exist_ok=True)
print('Notebook v6 |', torch.__version__, '|', torch.cuda.get_device_name(), flush=True)


## Model dependencies (not the trainer)

In [ ]:
from pathlib import Path

SHA = "22d3a86"  # bump to update code
BASE = f"https://raw.githubusercontent.com/jarenm1/triage/{SHA}"

FILES = [
    "rl/__init__.py",
    "rl/visual_encoder.py",
    "rl/venc_augment.py",
    "rl/venc_domain.py",
    "rl/vendor/__init__.py",
    "rl/vendor/lingbot_vision/__init__.py",
    "rl/vendor/lingbot_vision/attention.py",
    "rl/vendor/lingbot_vision/build.py",
    "rl/vendor/lingbot_vision/layers.py",
    "rl/vendor/lingbot_vision/loader.py",
    "rl/vendor/lingbot_vision/preprocess.py",
    "rl/vendor/lingbot_vision/vit.py",
    "rl/vendor/lingbot_vision/configs/lbot_vision_vitb.yaml",
]

for f in FILES:
    dest = ROOT / f
    dest.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(f"{BASE}/{f}", dest)
sys.path.insert(0, str(ROOT))
print('Model source revision:', SHA, flush=True)


## Configuration

In [ ]:
DATA = ROOT / 'data'
# Checkpoints and metrics go directly to Drive, not ephemeral runtime storage.
OUTPUT = Path('/content/drive/MyDrive/triage-results/m-256-colab.json')
args = SimpleNamespace(
    variant='m', teacher='lingbot', teacher_variant='base', teacher_ckpt=None,
    distill_dim=0, checkpoint=None, device='cuda', seed=0,
    steps=4000, batch=2, window=9, lr=3e-4,
    consistency_coef=0.5, distill_coef=0.5, domain_coef=0.1,
    weak_disc=False, augment=False,
    data=DATA/'train-256-rollouts.pt', val=DATA/'val-256-rollouts.pt',
    real=DATA/'real-256.pt', output=OUTPUT,
)
EVAL_EVERY = 200
EVAL_WINDOWS = 512
EVAL_BATCH = 2


## Mount Drive and stage data on local disk

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA.mkdir(parents=True, exist_ok=True)
for dest in (args.data, args.val, args.real):
    source = Path('/content/drive/MyDrive/triage-data') / dest.name
    if not source.is_file():
        raise FileNotFoundError(source)
    print('Staging', source.name, flush=True)
    if not dest.exists() or dest.stat().st_size != source.stat().st_size:
        shutil.copyfile(source, dest)
    print(dest, dest.stat().st_size, 'bytes', flush=True)
OUTPUT.parent.mkdir(parents=True, exist_ok=True)


## Training definitions
Packed data is memory-mapped on local disk. JPEG decoding is CPU, per selected batch, with no decoded-frame cache. The student and teacher compute on GPU. This favors bounded memory over unverified GPU-decoder assumptions.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

from rl.visual_encoder import VisualEncoder, VisualEncoderConfig
from rl.venc_augment import augment
from rl.venc_domain import (
    DomainDiscriminator,
    degrade,
    grad_reverse,
    load_teacher,
)

CONTROL_DT = 0.05
PROBE_SCALE = torch.tensor([4.5, 10.0, 0.75, 0.75])  # gap_x, wall_d, vx, vz
FUTURE_DIM = 64 * 48  # latent_channels * 6 * 8
DINO_MEAN = torch.tensor([0.485, 0.456, 0.406])
DINO_STD = torch.tensor([0.229, 0.224, 0.225])

# cfg = VisualEncoderConfig kwargs; consistency/domain/distill are trainer flags
VARIANTS = {
    "a": dict(cfg=dict(spatial_probe=False, temporal=False, warp=False,
                       motion_condition=False, diff_signal=False,
                       future_head=False)),
    "b": dict(cfg=dict(temporal=False, warp=False, motion_condition=False,
                       diff_signal=False, future_head=False)),
    "c": dict(cfg=dict(warp=False, motion_condition=False, diff_signal=False,
                       future_head=False)),
    "d": dict(cfg=dict(warp=False, diff_signal=False, future_head=False)),
    "e": dict(cfg=dict(diff_signal=False, future_head=False)),
    "f": dict(cfg=dict(future_head=False)),
    "g": dict(cfg=dict()),
    "h": dict(cfg=dict(global_temporal=True)),
    "i": dict(cfg=dict(), consistency=True),
    "j": dict(cfg=dict(distill=True), consistency=True, distill=True),
    "k": dict(cfg=dict(), consistency=True, domain=True),
    "l": dict(cfg=dict(distill=True), consistency=True, domain=True,
              distill=True),
    # M: distill on sim+real (positive alignment), disc is eval-only metric
    "m": dict(cfg=dict(distill=True), consistency=True, distill=True,
              distill_real=True, domain_eval=True),
}


class Teacher:
    """Frozen DINOv2/v3 -> patch token map (B,C,h/p,w/p)."""

    def __init__(self, name, device, checkpoint=None, variant="large"):
        self.net, self.patch, self.embed_dim = load_teacher(
            name, device, checkpoint, variant=variant
        )
        self.device = device
        self.mean = DINO_MEAN.reshape(1, 3, 1, 1).to(device)
        self.std = DINO_STD.reshape(1, 3, 1, 1).to(device)

    @torch.no_grad()
    def features(self, frames, chunk=32):
        outs = []
        for i in range(0, frames.shape[0], chunk):
            x = frames[i : i + chunk]
            x = x.float() / 255.0 if x.dtype == torch.uint8 else x
            x = (x - self.mean) / self.std
            h, w = x.shape[2] // self.patch * self.patch, \
                x.shape[3] // self.patch * self.patch
            x = F.interpolate(x, (h, w), mode="bilinear", align_corners=False)
            out = self.net.forward_features(x)["x_prenorm"]
            out = out[:, 1 + getattr(self.net, "n_storage_tokens", 0):]
            gh, gw = h // self.patch, w // self.patch
            outs.append(
                out.reshape(x.shape[0], gh, gw, -1).permute(0, 3, 1, 2)
            )
        return torch.cat(outs)


In [ ]:
class RolloutData:
    """Flat rollout storage -> contiguous per-env segments -> windows."""

    def __init__(self, path, device):
        d = torch.load(path, weights_only=False, mmap=True)
        self.device = device
        # frames stored as JPEG byte blobs (bounded RAM, Drive-friendly size)
        self.frames_jpeg = d.get("frames_jpeg")
        self.frames_len = d.get("frames_len")
        self.frames = d.get("frames")            # legacy raw uint8
        self.pframes_jpeg = d.get("pframes_jpeg")
        self.pframes_len = d.get("pframes_len")
        self.pframes = d.get("pframes")
        self.poses = d["poses"]            # (N,4)
        self.actions = d["actions"]        # (N,2)
        self.gap_x = d["gap_x"]
        self.wall_d = d["wall_d"]
        self.occ = d["occ"]                # (N,32)
        self.done = d["done"]
        self.env_id = d["env_id"]
        n = self.poses.shape[0]
        seg_start = torch.ones(n, dtype=torch.bool)
        seg_start[1:] = (self.env_id[1:] != self.env_id[:-1]) | self.done[:-1]
        self.seg_starts = seg_start.nonzero().flatten().tolist() + [n]

    def _decode(self, packed, lens, s, e):
        """Decode only requested JPEG rows to CPU uint8."""
        import io
        from PIL import Image
        return torch.stack([
            torch.from_numpy(
                np.array(Image.open(io.BytesIO(bytes(packed[i, : int(lens[i])].numpy()))))
            ).permute(2, 0, 1)
            for i in range(s, e)
        ])

    def _frames(self, s, e):
        if self.frames_jpeg is not None:
            return self._decode(self.frames_jpeg, self.frames_len, s, e)
        return self.frames[s:e]

    def _pframes(self, s, e):
        if self.pframes_jpeg is not None:
            return self._decode(self.pframes_jpeg, self.pframes_len, s, e)
        return self.pframes[s:e]

    def windows(self, length):
        out = []
        for i in range(len(self.seg_starts) - 1):
            s, e = self.seg_starts[i], self.seg_starts[i + 1]
            for w in range(s, e - length + 1):
                out.append((w, w + length))
        return out

    def batch(self, window_list, idx):
        sel = [window_list[i] for i in idx]
        frames = torch.stack([self._frames(s, e) for s, e in sel])
        poses = torch.stack([self.poses[s:e] for s, e in sel])
        actions = torch.stack([self.actions[s:e] for s, e in sel])
        gap = torch.stack([self.gap_x[s:e] for s, e in sel])
        wall = torch.stack([self.wall_d[s:e] for s, e in sel])
        occ = torch.stack([self.occ[s:e] for s, e in sel])
        dx = torch.diff(poses[..., 0], dim=-1, prepend=poses[:, :1, 0])
        dz = torch.diff(poses[..., 1], dim=-1, prepend=poses[:, :1, 1])
        dx[:, 0] = poses[:, 0, 2] * CONTROL_DT
        dz[:, 0] = poses[:, 0, 3] * CONTROL_DT
        motion = torch.stack(
            [dx, dz, poses[..., 2], poses[..., 3],
             torch.full_like(dx, CONTROL_DT)],
            dim=-1,
        )
        probe_target = torch.stack(
            [gap, wall, poses[..., 2], poses[..., 3]], dim=-1
        ) / PROBE_SCALE
        out = {
            "frames": frames.to(self.device),
            "motion": motion.to(self.device),
            "actions": actions.to(self.device),
            "probe": probe_target.to(self.device),
            "occ": occ.to(self.device),
        }
        if self.pframes is not None or self.pframes_jpeg is not None:
            out["pframes"] = torch.stack(
                [self._pframes(s, e) for s, e in sel]
            ).to(self.device)
        return out

class _JpegFrames:
    """Lazy view over packed JPEG rows; decodes selected rows to CPU uint8."""

    def __init__(self, packed, lens, device):
        self.packed, self.lens, self.device = packed, lens, device
        self.shape = (packed.shape[0],)  # callers only need shape[0]

    def __len__(self):
        return self.packed.shape[0]

    def __getitem__(self, idx):
        import io
        from PIL import Image
        if isinstance(idx, int):
            idx = [idx]
        if isinstance(idx, torch.Tensor):
            idx = idx.tolist()
        if isinstance(idx, slice):
            idx = list(range(*idx.indices(self.packed.shape[0])))
        out = []
        for i in idx:
            n = int(self.lens[i])
            img = Image.open(io.BytesIO(bytes(self.packed[i, :n].numpy())))
            out.append(torch.from_numpy(np.array(img)).permute(2, 0, 1))
        return torch.stack(out)


def future_target(enc, frames):
    with torch.no_grad():
        f, _ = enc.encode(frames)
        f = F.adaptive_avg_pool2d(f, (6, 8))
        return F.normalize(f.reshape(f.shape[0], -1), dim=-1)


def paired_cosine(enc, frames, pframes):
    with torch.no_grad():
        f1, _ = enc.encode(frames)
        f2, _ = enc.encode(pframes)
        return F.cosine_similarity(
            f1.reshape(f1.shape[0], -1), f2.reshape(f2.shape[0], -1), dim=-1
        ).mean()


def evaluate(enc, data, windows, batch=64, device="cuda", disc=None,
             real_frames=None):
    enc.eval()
    scale = PROBE_SCALE.to(device)
    mae = torch.zeros(4, device=device)
    iou_n, iou_d = 0.0, 0.0
    fut_n, fut_c = 0.0, 0
    cos_n, cos_c = 0.0, 0
    dom_correct, dom_n = 0, 0
    n = 0
    with torch.no_grad():
        for i in range(0, len(windows), batch):
            print(f"Validation windows {i}/{len(windows)}", flush=True)
            b = data.batch(windows, list(range(i, min(i + batch, len(windows)))))
            outs, _ = enc.forward_sequence(
                b["frames"][:, :-1], b["motion"][:, :-1], b["actions"][:, :-1]
            )
            pred = outs["probe"] * scale
            tgt = b["probe"][:, :-1] * scale
            mae += (pred - tgt).abs().sum(dim=(0, 1))
            p_occ = (outs["occupancy"] > 0).float()
            t_occ = b["occ"][:, :-1]
            iou_n += (p_occ * t_occ).sum().item()
            iou_d += ((p_occ + t_occ) > 0).float().sum().item()
            if "future_f" in outs:
                tgt_f = future_target(enc, b["frames"][:, -1])
                pred_f = F.normalize(
                    outs["future_f"][:, -1].reshape(-1, FUTURE_DIM), dim=-1
                )
                fut_n += F.mse_loss(pred_f, tgt_f, reduction="sum").item()
                fut_c += pred_f.shape[0]
            if "pframes" in b:
                cos_n += paired_cosine(
                    enc,
                    b["frames"][:, :-1].reshape(-1, *b["frames"].shape[2:]),
                    b["pframes"][:, :-1].reshape(-1, *b["pframes"].shape[2:]),
                ).item() * b["frames"].shape[0]
                cos_c += b["frames"].shape[0]
            if disc is not None:
                f_sim = outs["f"].reshape(-1, *outs["f"].shape[2:])
                dom_correct += (disc(f_sim) < 0).sum().item()
                dom_n += f_sim.shape[0]
            n += b["frames"].shape[0] * (b["frames"].shape[1] - 1)
        if disc is not None and real_frames is not None:
            for j in range(0, real_frames.shape[0], batch):
                if j % (batch * 25) == 0:
                    print(f"Validation real frames {j}/{real_frames.shape[0]}", flush=True)
                f_real, _ = enc.encode(real_frames[j : j + batch].to(device))
                dom_correct += (disc(f_real) > 0).sum().item()
                dom_n += f_real.shape[0]
    enc.train()
    return {
        "mae_gap_x": (mae[0] / n).item(),
        "mae_wall_d": (mae[1] / n).item(),
        "mae_vx": (mae[2] / n).item(),
        "mae_vz": (mae[3] / n).item(),
        "occ_iou": iou_n / max(iou_d, 1e-9),
        "future_mse": fut_n / max(fut_c, 1),
        "paired_cos": cos_n / max(cos_c, 1),
        "domain_acc": dom_correct / max(dom_n, 1),
    }


## Load data — no eager full-dataset decoding

In [ ]:
device = torch.device(args.device)
print('Mapping real frames', flush=True)
real_frames = None
if args.real:
    rd = torch.load(args.real, weights_only=False, mmap=True)
    if "frames_jpeg" in rd:
        real_frames = _JpegFrames(rd["frames_jpeg"], rd["frames_len"], device)
    else:
        real_frames = rd["frames"]

train = RolloutData(args.data, device)
val = RolloutData(args.val, device)
train_windows = train.windows(args.window)
val_windows = val.windows(args.window)
print(f"train windows: {len(train_windows)}, val: {len(val_windows)}")

if not train_windows or not val_windows:
    raise ValueError('No contiguous windows; check data and window length.')
print('Data ready', flush=True)


## Initialize teacher, student and optimizers

In [ ]:
print('Initializing teacher (weights may download)', flush=True)
torch.manual_seed(args.seed)
device = torch.device(args.device)
spec = VARIANTS[args.variant]
cfg = VisualEncoderConfig(**spec["cfg"])
teacher = (
    Teacher(args.teacher, device, args.teacher_ckpt,
            variant=args.teacher_variant)
    if spec.get("distill")
    else None
)
if teacher is not None:
    cfg.distill_dim = args.distill_dim or teacher.embed_dim

enc = VisualEncoder(cfg).to(device)
if args.checkpoint:
    enc.load_state_dict(torch.load(args.checkpoint, weights_only=True))
disc = (
    DomainDiscriminator(cfg.latent_channels, weak=args.weak_disc).to(device)
    if spec.get("domain") or spec.get("domain_eval")
    else None
)
disc_opt = (
    torch.optim.AdamW(disc.parameters(), lr=1e-3)
    if disc is not None and spec.get("domain_eval")
    else None
)
params = list(enc.parameters()) + (
    list(disc.parameters()) if disc is not None and spec.get("domain") else []
)
opt = torch.optim.AdamW(params, lr=args.lr, weight_decay=1e-4)
log = []
next_step = 0
print('Models and optimizers ready', flush=True)


## Train in this kernel
Each step prints before loading and after the update. Evaluation is explicit and logged. Interrupting this cell leaves the model and optimizer inspectable; rerunning continues from `next_step` in this runtime. Saved `.pt` files are encoder weights for downstream use, not full optimizer-resume snapshots.

In [ ]:
for step in range(next_step, args.steps):
    print(f"Step {step + 1}/{args.steps}: loading batch", flush=True)
    started = time.monotonic()
    enc.train()
    idx = torch.randint(0, len(train_windows), (args.batch,)).tolist()
    b = train.batch(train_windows, idx)
    print("Forward / loss / backward", flush=True)
    frames_in = b["frames"][:, :-1]
    if args.augment:
        frames_in = augment(
            frames_in.reshape(-1, *frames_in.shape[2:])
        ).reshape(frames_in.shape)
    outs, _ = enc.forward_sequence(
        frames_in, b["motion"][:, :-1], b["actions"][:, :-1]
    )
    loss_probe = F.mse_loss(outs["probe"], b["probe"][:, :-1])
    loss_occ = F.binary_cross_entropy_with_logits(
        outs["occupancy"], b["occ"][:, :-1]
    )
    loss = loss_probe + loss_occ
    logs = {"probe": loss_probe.item(), "occ": loss_occ.item()}

    if "future_f" in outs:
        tgt_f = future_target(enc, b["frames"][:, -1])
        pred_f = F.normalize(
            outs["future_f"][:, -1].reshape(-1, FUTURE_DIM), dim=-1
        )
        loss_fut = F.mse_loss(pred_f, tgt_f)
        loss = loss + 0.5 * loss_fut
        logs["future"] = loss_fut.item()

    if spec.get("consistency") and "pframes" in b:
        flat_p = b["pframes"][:, :-1].reshape(-1, *b["pframes"].shape[2:])
        f_p, _ = enc.encode(flat_p)
        f_t = outs["f"].reshape(-1, *outs["f"].shape[2:])
        loss_con = (
            1
            - F.cosine_similarity(
                f_t.reshape(f_t.shape[0], -1),
                f_p.reshape(f_p.shape[0], -1),
                dim=-1,
            ).mean()
        )
        loss = loss + args.consistency_coef * loss_con
        logs["consist"] = loss_con.item()

    if disc is not None and real_frames is not None and spec.get("domain"):
        # sim features vs real features; GRL reverses into enc
        f_sim = outs["f"].reshape(-1, *outs["f"].shape[2:])
        n_real = min(f_sim.shape[0], real_frames.shape[0])
        ridx = torch.randint(0, real_frames.shape[0], (n_real,))
        f_real, _ = enc.encode(real_frames[ridx].to(device))
        logits = torch.cat(
            [
                disc(grad_reverse(f_sim[:n_real])),
                disc(grad_reverse(f_real)),
            ]
        )
        labels = torch.cat(
            [torch.zeros(n_real, device=device),
             torch.ones(n_real, device=device)]
        )
        loss_dom = F.binary_cross_entropy_with_logits(logits, labels)
        loss = loss + args.domain_coef * loss_dom
        logs["domain"] = loss_dom.item()

    if disc is not None and real_frames is not None and spec.get("domain_eval"):
        # eval-only disc: train to distinguish, no gradient into enc.
        # domain_acc ~0.5 then genuinely means the latent is invariant.
        f_sim = outs["f"].reshape(-1, *outs["f"].shape[2:]).detach()
        n_real = min(f_sim.shape[0], real_frames.shape[0])
        ridx = torch.randint(0, real_frames.shape[0], (n_real,))
        with torch.no_grad():
            f_real, _ = enc.encode(real_frames[ridx].to(device))
        logits = torch.cat([disc(f_sim[:n_real]), disc(f_real)])
        labels = torch.cat(
            [torch.zeros(n_real, device=device),
             torch.ones(n_real, device=device)]
        )
        loss_disc = F.binary_cross_entropy_with_logits(logits, labels)
        disc_opt.zero_grad()
        loss_disc.backward()
        disc_opt.step()
        logs["domain"] = loss_disc.item()

    if teacher is not None and "distill_f" in outs:
        flat_f = b["frames"][:, :-1].reshape(-1, *b["frames"].shape[2:])
        s_feat = outs["distill_f"].reshape(-1, *outs["distill_f"].shape[2:])
        if spec.get("distill_real") and real_frames is not None:
            # distill on real frames too: positive domain alignment
            n_real = min(flat_f.shape[0], real_frames.shape[0])
            ridx = torch.randint(0, real_frames.shape[0], (n_real,))
            r_f, _ = enc.encode(real_frames[ridx].to(device))
            r_feat = enc.distill_proj(r_f)
            flat_f = torch.cat([flat_f, real_frames[ridx].to(device)])
            s_feat = torch.cat([s_feat, r_feat])

        print("Teacher features", flush=True)
        t_feat = teacher.features(flat_f, chunk=4)
        t_feat = F.interpolate(
            t_feat, s_feat.shape[2:], mode="bilinear", align_corners=False
        )
        loss_dis = (
            1
            - F.cosine_similarity(
                F.normalize(s_feat, dim=1).reshape(s_feat.shape[0], -1),
                F.normalize(t_feat, dim=1).reshape(t_feat.shape[0], -1),
                dim=-1,
            ).mean()
        )
        loss = loss + args.distill_coef * loss_dis
        logs["distill"] = loss_dis.item()

    opt.zero_grad()
    loss.backward()
    opt.step()
    next_step = step + 1
    print(f"Step {next_step}: loss={loss.item():.4f}, {time.monotonic()-started:.1f}s, "
          f"CUDA allocated={torch.cuda.memory_allocated()/2**30:.2f} GiB", logs, flush=True)
    # Release completed autograd graphs before validation or the next step.
    del loss, outs, b
    if next_step % EVAL_EVERY == 0 or next_step == args.steps:
        print('Saving encoder before validation', flush=True)
        torch.save(enc.state_dict(), OUTPUT.with_suffix('.pt'))
        print('Evaluating', flush=True)
        metrics = evaluate(enc, val, val_windows[:EVAL_WINDOWS], batch=EVAL_BATCH,
                           device=device, disc=disc, real_frames=real_frames)
        metrics.update(step=next_step, **logs)
        log.append(metrics)
        OUTPUT.write_text(json.dumps({'variant': args.variant, 'config': spec,
                                     'final': metrics, 'history': log}, indent=2))
        print(metrics, flush=True)
print('Training complete:', OUTPUT, flush=True)


## Save current weights and inspect results
Run after an interrupt as well as after completion. Copy the `.pt` encoder checkpoint back for local RL integration.

In [ ]:
torch.save(enc.state_dict(), OUTPUT.with_suffix('.pt'))
print('Encoder weights:', OUTPUT.with_suffix('.pt'), flush=True)
if OUTPUT.exists():
    print(json.dumps(json.loads(OUTPUT.read_text()), indent=2))
else:
    print('No completed validation yet; current encoder weights saved.')
